In [ ]:
import numpy as np

# Reproducible random data
rng = np.random.default_rng(42)

In [ ]:
# Generate synthetic animal data
def generate_animals(n):
  dogs = np.column_stack([
      rng.uniform(8, 30, n), # Weight
      rng.uniform(30, 65, n), # Height
      rng.binomial(1, 0.10, n) # Long ears
  ])

  cats = np.column_stack([
    rng.uniform(2.5, 7, n),
    rng.uniform(20, 35, n),
    rng.binomial(1, 0.05, n)
  ])

  rabbits = np.column_stack([
    rng.uniform(1, 4, n),
    rng.uniform(12, 25, n),
    rng.binomial(1, 0.95, n)
  ])

  X = np.vstack([dogs, cats, rabbits])

  # 0 = Dog, 1 = Cat, 2 = Rabbit
  y = np.array([0] * n + [1] * n + [2] * n)

  return X, y

In [ ]:
X, y = generate_animals(100)

indices = rng.permutation(len(X))

X = X[indices]
y = y[indices]

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
X_train = X_train / np.array([30, 65, 1])
x_test = X_test / np.array([30, 65, 1])

In [ ]:
y_train_one_hot = np.eye(3)[y_train]

In [ ]:
input_size, hidden_size, output_size = 3, 5, 3

In [ ]:
W1 = rng.normal(
    0, np.sqrt(1 / input_size),
    (input_size, hidden_size)
)

b1 = np.zeros((1, hidden_size))

W2 = rng.normal(
    0, np.sqrt(1 / hidden_size),
    (hidden_size, output_size)
)

b2 = np.zeros((1, output_size))

learning_rate = 0.5
epochs = 2000

In [ ]:
def tanh(x):
  return np.tanh(x)

def softmax(x):
  exp = np.exp(x - np.max(x, axis=1, keepdims=True))
  return exp / np.sum(exp, axis=1, keepdims=True)

In [ ]:
for epoch in range(epochs):
  z1 = X_train @ W1 + b1
  a1 = tanh(z1)

  z2 = a1 @ W2 + b2
  predictions = softmax(z2)

  loss = -np.mean(
      np.sum(y_train_one_hot * np.log(predictions + 1e-9), axis=1)
  )

  m = len(X_train)

  dz2 = (predictions - y_train_one_hot) / m

  dW2 = a1.T @ dz2
  db2 = np.sum(dz2, axis=0, keepdims=True)

  dz1 = (dz2 @ W2.T) * (1 - a1 ** 2)

  dW1 = X_train.T @ dz1
  db1 = np.sum(dz1, axis=0, keepdims=True)

  W2 -= learning_rate * dW2
  b2 -= learning_rate * db2

  W1 -= learning_rate * dW1
  b1 -= learning_rate * db1

  if epoch % 200 == 0:
    print(f"Epoch: {epoch:4d} | Loss: {loss:.4f}")

Epoch:    0 | Loss: 0.5682
Epoch:  200 | Loss: 0.5583
Epoch:  400 | Loss: 0.5486
Epoch:  600 | Loss: 0.5391
Epoch:  800 | Loss: 0.5298
Epoch: 1000 | Loss: 0.5206
Epoch: 1200 | Loss: 0.5117
Epoch: 1400 | Loss: 0.5029
Epoch: 1600 | Loss: 0.4943
Epoch: 1800 | Loss: 0.4859


In [ ]:
z1 = X_test @ W1 + b1
a1 = tanh(z1)

z2 = a1 @ W2 + b2
test_predictions = softmax(z2)

predicted_classes = np.argmax(test_predictions, axis=1)

accuracy = np.mean(predicted_classes == y_test)

print("\nTraining completed!")
print(f"Test accuracy: {accuracy * 100:.2f}%")


Training completed!
Test accuracy: 33.33%


In [ ]:
from binascii import b2a_qp
new_animal = np.array([[3.0, 18, 1]])

new_animal = new_animal / np.array([30, 65, 1])

z1 = new_animal @ W1 + b1
a1 = tanh(z1)

z2 = a1 @ W2 + b2
probabilities = softmax(z2)

classes = ["Dog", "Cat", "Rabbit"]

print("\nClassification result:")

for name, probability, in zip(classes, probabilities[0]):
  print(f"{name}: {probability * 100:.2f}%")

predicted_class = np.argmax(probabilities)

print(f"\nPredicted animal: {classes[predicted_class]}")


Classification result:
Dog: 100.00%
Cat: 0.00%
Rabbit: 0.00%

Predicted animal: Dog
